# Silver → Gold

Modelo dimensional (Star Schema) para análise dos preços.

- `dim_produto` e `dim_calendario`: dimensões simples, carregadas com MERGE
- `dim_posto`: histórico de bandeira e nome de cada posto (SCD Tipo 2)
- `fato_precos`: um registro por posto, produto e dia, ligado à versão do posto válida na data da coleta
- `vw_precos`: fato com as dimensões e o preço corrigido pelo IPCA

In [0]:
from datetime import date
from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalogo = "preco_na_bomba"
silver = f"{catalogo}.silver"
gold = f"{catalogo}.gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold}")

df_precos = spark.table(f"{silver}.tb_precos")

## 1. dim_produto

Chave substituta gerada pelo Delta (`IDENTITY`). O MERGE só insere produtos que ainda não existem.

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {gold}.dim_produto (
        sk_produto BIGINT GENERATED ALWAYS AS IDENTITY,
        produto STRING,
        unidade_medida STRING
    )
""")

df_precos.select("produto", "unidade_medida").distinct().createOrReplaceTempView("produtos_silver")

spark.sql(f"""
    MERGE INTO {gold}.dim_produto AS t
    USING produtos_silver AS s
    ON t.produto = s.produto
    WHEN NOT MATCHED THEN INSERT (produto, unidade_medida) VALUES (s.produto, s.unidade_medida)
""")

display(spark.table(f"{gold}.dim_produto").orderBy("sk_produto"))

sk_produto,produto,unidade_medida
1,Gasolina Aditivada,R$ / litro
2,GNV,R$ / m³
3,Diesel S10,R$ / litro
4,Diesel,R$ / litro
5,Etanol,R$ / litro
6,Gasolina,R$ / litro


## 2. dim_calendario

Um dia por linha, do primeiro ao último ano com coletas. A chave `sk_data` é a própria data no formato `yyyyMMdd`.

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {gold}.dim_calendario (
        sk_data INT,
        data DATE,
        ano INT,
        semestre INT,
        trimestre INT,
        mes INT,
        ano_mes STRING,
        dia_semana INT,
        fim_de_semana BOOLEAN
    )
""")

inicio, fim = df_precos.agg(F.min("data_coleta"), F.max("data_coleta")).first()

(
    spark.sql(f"""
        SELECT explode(sequence(
            trunc(DATE'{inicio}', 'year'),
            make_date(year(DATE'{fim}'), 12, 31),
            INTERVAL 1 DAY
        )) AS data
    """)
    .select(
        F.date_format("data", "yyyyMMdd").cast("int").alias("sk_data"),
        "data",
        F.year("data").alias("ano"),
        F.when(F.month("data") <= 6, 1).otherwise(2).alias("semestre"),
        F.quarter("data").alias("trimestre"),
        F.month("data").alias("mes"),
        F.date_format("data", "yyyy-MM").alias("ano_mes"),
        F.dayofweek("data").alias("dia_semana"),
        F.dayofweek("data").isin(1, 7).alias("fim_de_semana"),
    )
    .createOrReplaceTempView("calendario_novo")
)

spark.sql(f"""
    MERGE INTO {gold}.dim_calendario AS t
    USING calendario_novo AS s
    ON t.sk_data = s.sk_data
    WHEN NOT MATCHED THEN INSERT *
""")

display(spark.sql(f"SELECT MIN(data), MAX(data), COUNT(*) AS dias FROM {gold}.dim_calendario"))

MIN(data),MAX(data),dias
2024-01-01,2025-12-31,731
